# Muse S リアルタイムEEGパイプライン

**事前準備**: 別ターミナルで `muselsl stream` を起動しておくこと

```
C:\Users\hiro2\anaconda3\Scripts\muselsl.exe stream
```

チャンネル順: `[TP9, AF7, AF8, TP10]` (index: 0, 1, 2, 3)

## Cell 1: LSLストリームからEEGデータを受信する基本テスト

In [ ]:
from pylsl import StreamInlet, resolve_byprop
import numpy as np

# --- 定数 ---
CH_NAMES = ['TP9', 'AF7', 'AF8', 'TP10']
FS = 256  # Muse S サンプリング周波数 [Hz]
N_TEST_SECONDS = 3
N_TEST_SAMPLES = FS * N_TEST_SECONDS

# LSLストリームを検索（タイムアウト10秒）
print('LSLストリームを検索中...')
streams = resolve_byprop('type', 'EEG', timeout=10)

if not streams:
    raise RuntimeError('LSLストリームが見つかりません。'
                       '別ターミナルで muselsl stream を起動してください。')

inlet = StreamInlet(streams[0], max_buflen=30)
info = inlet.info()
print(f'接続成功: {info.name()} ({info.channel_count()}ch, {info.nominal_srate():.0f}Hz)')

# --- N_TEST_SECONDS 秒分のデータを受信 ---
print(f'\n{N_TEST_SECONDS}秒間データを受信中...')
samples, timestamps = [], []

while len(samples) < N_TEST_SAMPLES:
    chunk, ts = inlet.pull_chunk(timeout=1.0, max_samples=32)
    if chunk:
        samples.extend(chunk)
        timestamps.extend(ts)

data = np.array(samples)[:N_TEST_SAMPLES]   # shape: (samples, 4)
t    = np.array(timestamps)[:N_TEST_SAMPLES]

print(f'\n受信データ shape: {data.shape}  (samples x channels)')
print(f'時間範囲: {t[-1]-t[0]:.2f}s')
print(f'実効サンプリングレート: {(len(t)-1)/(t[-1]-t[0]):.1f} Hz')
print('\n各チャンネル統計 (μV):')
for i, ch in enumerate(CH_NAMES):
    print(f'  {ch}: mean={data[:,i].mean():.1f}  std={data[:,i].std():.1f}')

print('\nCell 1 完了 ✓')

## Cell 2: バッファを使ったリアルタイムバンドパワー（θ/α/β）計算

In [ ]:
from scipy.signal import welch
from collections import deque
import time

# --- バンド定義 ---
BANDS = {
    'theta': (4, 8),
    'alpha': (8, 13),
    'beta':  (13, 30),
}

# --- バッファ設定 ---
WINDOW_SEC  = 2.0   # パワー計算ウィンドウ [s]
STEP_SEC    = 0.25  # 更新間隔 [s]
WINDOW_SAMP = int(FS * WINDOW_SEC)
STEP_SAMP   = int(FS * STEP_SEC)

def bandpower(data_ch: np.ndarray, fs: int, band: tuple) -> float:
    """1チャンネルの帯域パワーをWelch法で計算 (μV²)"""
    freqs, psd = welch(data_ch, fs=fs, nperseg=fs)  # 1s解像度
    idx = np.logical_and(freqs >= band[0], freqs <= band[1])
    return float(np.trapz(psd[idx], freqs[idx]))

def compute_emotions(powers: dict) -> dict:
    """
    Arousal  = β / α  (AF7+AF8 平均)
    Valence  = log(αR) - log(αL)  (AF8=right, AF7=left)
    """
    alpha_L = powers['alpha_AF7']
    alpha_R = powers['alpha_AF8']
    beta_L  = powers['beta_AF7']
    beta_R  = powers['beta_AF8']

    alpha_mean = (alpha_L + alpha_R) / 2 + 1e-9
    beta_mean  = (beta_L  + beta_R)  / 2 + 1e-9
    arousal = beta_mean / alpha_mean

    valence = np.log(alpha_R + 1e-9) - np.log(alpha_L + 1e-9)

    return {'arousal': arousal, 'valence': valence}

# --- リングバッファ (channel x samples) ---
buf = deque(maxlen=WINDOW_SAMP)  # 各要素: (4,)のnp.array
step_counter = 0
N_DISPLAY = 8  # 表示する更新回数

print('リアルタイムバンドパワー計算開始...')
print(f'ウィンドウ: {WINDOW_SEC}s / 更新間隔: {STEP_SEC}s')
print('-' * 65)
print(f'{"step":>5} | {"θ_AF7":>7} {"α_AF7":>7} {"β_AF7":>7} | '
      f'{"Arousal":>8} {"Valence":>8}')
print('-' * 65)

inlet.flush()  # バッファをクリア
accumulated = 0

while step_counter < N_DISPLAY:
    chunk, _ = inlet.pull_chunk(timeout=0.5, max_samples=STEP_SAMP)
    if not chunk:
        continue

    for s in chunk:
        buf.append(np.array(s))  # (4,)
    accumulated += len(chunk)

    # バッファが埋まるまでは計算しない
    if len(buf) < WINDOW_SAMP:
        continue

    # STEP_SAMP サンプル溜まるごとに計算
    if accumulated < STEP_SAMP:
        continue
    accumulated = 0

    mat = np.array(buf)  # (WINDOW_SAMP, 4)
    # CH: [TP9=0, AF7=1, AF8=2, TP10=3]
    powers = {
        f'{band}_AF7': bandpower(mat[:, 1], FS, rng)
        for band, rng in BANDS.items()
    }
    powers.update({
        f'{band}_AF8': bandpower(mat[:, 2], FS, rng)
        for band, rng in BANDS.items()
    })

    emo = compute_emotions(powers)

    print(f'{step_counter+1:>5} | '
          f'{powers["theta_AF7"]:>7.2f} '
          f'{powers["alpha_AF7"]:>7.2f} '
          f'{powers["beta_AF7"]:>7.2f} | '
          f'{emo["arousal"]:>8.3f} '
          f'{emo["valence"]:>8.3f}')

    step_counter += 1

print('-' * 65)
print('Cell 2 完了 ✓')

## Cell 3: matplotlib リアルタイム可視化

- 左上: 生波形 (AF7 / AF8)
- 右上: PSD (θ/α/β 帯域を色分け)
- 左下: バンドパワー時系列
- 右下: Arousal / Valence メーター

**停止方法**: グラフウィンドウを閉じる

In [ ]:
import matplotlib
matplotlib.use('TkAgg')   # VS Code では TkAgg が安定
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.animation import FuncAnimation

# ---- 設定 ----
HISTORY_LEN = 40   # バンドパワー時系列の表示ステップ数

# ---- 共有状態 ----
raw_buf  = deque(maxlen=WINDOW_SAMP)
bp_history = {k: deque([0.0]*HISTORY_LEN, maxlen=HISTORY_LEN)
              for k in ('theta', 'alpha', 'beta')}
emo_history = {'arousal': deque([0.0]*HISTORY_LEN, maxlen=HISTORY_LEN),
               'valence': deque([0.0]*HISTORY_LEN, maxlen=HISTORY_LEN)}
current_powers = {'theta_AF7': 0, 'alpha_AF7': 0, 'beta_AF7': 0,
                  'alpha_AF8': 0, 'beta_AF8': 0}
step_acc = [0]

inlet.flush()

# ---- Figure 構築 ----
fig, axes = plt.subplots(2, 2, figsize=(12, 7))
fig.suptitle('Muse S リアルタイムEEGモニタ', fontsize=13)
ax_raw, ax_psd, ax_bp, ax_emo = axes[0,0], axes[0,1], axes[1,0], axes[1,1]

t_raw = np.linspace(-WINDOW_SEC, 0, WINDOW_SAMP)
t_hist = np.arange(-HISTORY_LEN, 0) * STEP_SEC

# 生波形
line_af7, = ax_raw.plot(t_raw, np.zeros(WINDOW_SAMP), lw=0.8, label='AF7', color='steelblue')
line_af8, = ax_raw.plot(t_raw, np.zeros(WINDOW_SAMP), lw=0.8, label='AF8', color='tomato', alpha=0.8)
ax_raw.set_xlim(-WINDOW_SEC, 0); ax_raw.set_ylim(-200, 200)
ax_raw.set_xlabel('time [s]'); ax_raw.set_ylabel('μV')
ax_raw.set_title('生波形 (AF7 / AF8)'); ax_raw.legend(loc='upper right')
ax_raw.axhline(0, color='k', lw=0.4)

# PSD
freqs_dummy = np.linspace(0, FS//2, FS//2+1)
line_psd_af7, = ax_psd.semilogy(freqs_dummy, np.ones_like(freqs_dummy),
                                 lw=1.2, label='AF7', color='steelblue')
line_psd_af8, = ax_psd.semilogy(freqs_dummy, np.ones_like(freqs_dummy),
                                 lw=1.2, label='AF8', color='tomato', alpha=0.8)
for c, (lo, hi) in [('cyan', BANDS['theta']), ('lime', BANDS['alpha']),
                    ('orange', BANDS['beta'])]:
    ax_psd.axvspan(lo, hi, alpha=0.15, color=c)
ax_psd.set_xlim(0, 40); ax_psd.set_xlabel('Hz'); ax_psd.set_ylabel('PSD (μV²/Hz)')
ax_psd.set_title('パワースペクトル密度'); ax_psd.legend(loc='upper right')
patches = [mpatches.Patch(color='cyan', alpha=0.4, label='θ'),
           mpatches.Patch(color='lime', alpha=0.4, label='α'),
           mpatches.Patch(color='orange', alpha=0.4, label='β')]
ax_psd.legend(handles=patches + [line_psd_af7, line_psd_af8], loc='upper right', fontsize=7)

# バンドパワー時系列
lines_bp = {
    'theta': ax_bp.plot(t_hist, list(bp_history['theta']), lw=1.2, color='cyan',   label='θ')[0],
    'alpha': ax_bp.plot(t_hist, list(bp_history['alpha']), lw=1.2, color='lime',   label='α')[0],
    'beta':  ax_bp.plot(t_hist, list(bp_history['beta']),  lw=1.2, color='orange', label='β')[0],
}
ax_bp.set_xlim(t_hist[0], 0); ax_bp.set_ylim(0, 50)
ax_bp.set_xlabel(f'time [s]'); ax_bp.set_ylabel('power (μV²)')
ax_bp.set_title('バンドパワー時系列 (AF7)'); ax_bp.legend()

# Arousal / Valence
line_ar, = ax_emo.plot(t_hist, list(emo_history['arousal']), lw=1.5,
                        color='crimson', label='Arousal (β/α)')
line_va, = ax_emo.plot(t_hist, list(emo_history['valence']), lw=1.5,
                        color='mediumpurple', label='Valence (αR−αL)')
ax_emo.axhline(0, color='k', lw=0.5, ls='--')
ax_emo.set_xlim(t_hist[0], 0); ax_emo.set_ylim(-3, 5)
ax_emo.set_xlabel('time [s]')
ax_emo.set_title('感情指標'); ax_emo.legend()

plt.tight_layout()

# ---- アップデート関数 ----
def update(_):
    # LSLからチャンクを取得
    chunk, _ = inlet.pull_chunk(timeout=0.0, max_samples=32)
    if chunk:
        for s in chunk:
            raw_buf.append(np.array(s))
        step_acc[0] += len(chunk)

    if len(raw_buf) < WINDOW_SAMP:
        return

    mat = np.array(raw_buf)  # (WINDOW_SAMP, 4)

    # 生波形更新
    line_af7.set_ydata(mat[:, 1])
    line_af8.set_ydata(mat[:, 2])
    # y軸オートスケール（外れ値対策: 1-99パーセンタイル）
    vmin = np.percentile(mat[:, 1:3], 1)
    vmax = np.percentile(mat[:, 1:3], 99)
    margin = max((vmax - vmin) * 0.2, 10)
    ax_raw.set_ylim(vmin - margin, vmax + margin)

    # PSD 更新
    f7, p7 = welch(mat[:, 1], fs=FS, nperseg=FS)
    f8, p8 = welch(mat[:, 2], fs=FS, nperseg=FS)
    line_psd_af7.set_data(f7, np.maximum(p7, 1e-6))
    line_psd_af8.set_data(f8, np.maximum(p8, 1e-6))

    # STEP_SAMP ごとにバンドパワー計算
    if step_acc[0] >= STEP_SAMP:
        step_acc[0] = 0

        pw = {}
        for band, rng in BANDS.items():
            pw[f'{band}_AF7'] = bandpower(mat[:, 1], FS, rng)
            pw[f'{band}_AF8'] = bandpower(mat[:, 2], FS, rng)

        for band in ('theta', 'alpha', 'beta'):
            bp_history[band].append(pw[f'{band}_AF7'])

        emo = compute_emotions(pw)
        emo_history['arousal'].append(emo['arousal'])
        emo_history['valence'].append(emo['valence'])

        # バンドパワー時系列更新
        for band, line in lines_bp.items():
            line.set_ydata(list(bp_history[band]))
        bp_max = max(max(bp_history[b]) for b in ('theta','alpha','beta')) * 1.2
        ax_bp.set_ylim(0, max(bp_max, 1))

        # 感情指標更新
        line_ar.set_ydata(list(emo_history['arousal']))
        line_va.set_ydata(list(emo_history['valence']))
        ar_vals = list(emo_history['arousal'])
        va_vals = list(emo_history['valence'])
        y_lo = min(min(va_vals) - 0.5, -0.5)
        y_hi = max(max(ar_vals) + 0.5, 2.0)
        ax_emo.set_ylim(y_lo, y_hi)

    return (line_af7, line_af8, line_psd_af7, line_psd_af8,
            *lines_bp.values(), line_ar, line_va)

# ---- アニメーション開始 ----
ani = FuncAnimation(fig, update, interval=100, blit=False, cache_frame_data=False)
plt.show()
print('可視化ウィンドウが閉じられました。Cell 3 完了 ✓')